# Transkribus LLM Tagging

This notebook downloads a PAGE XML transcript from Transkribus, sends it to an LLM for processing, and uploads the result back.

**Steps:**
1. Install dependencies
2. Configure your credentials
3. Download PAGE XML from Transkribus
4. Process with LLM
5. Upload result back to Transkribus

## 1. Install dependencies

In [1]:
!pip install generic-llm-api-client requests xmltodict

  Using cached generic_llm_api_client-0.3.10-py3-none-any.whl.metadata (16 kB)
  Using cached anthropic-0.84.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached openai-2.26.0-py3-none-any.whl.metadata (29 kB)
  Using cached google_genai-1.67.0-py3-none-any.whl.metadata (52 kB)
  Using cached cohere-5.20.7-py3-none-any.whl.metadata (3.6 kB)
  Using cached geonamescache-3.0.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached jiter-0.13.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached fastavro-1.12.1-cp312-cp312-win_amd64.whl.metadata (5.9 kB)
  Using cached pydantic_core-2.42.0-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached types


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuration

Fill in your Transkribus credentials, the document you want to process, and your LLM API key.

In [2]:
# --- Transkribus credentials ---
TRANSKRIBUS_USERNAME = ""
TRANSKRIBUS_PASSWORD = ""

# --- Document to process ---
COLLECTION_ID = 2205373      # e.g. 12345
DOCUMENT_ID   = 11509276      # e.g. 67890
PAGE_NR       = 55      # page number (1-based)

# --- LLM settings ---
# Supported providers: "openai", "anthropic", "google", "mistral", "deepseek", "openrouter"
LLM_PROVIDER  = "openai"
LLM_API_KEY   = ""          # API key provided by your instructor
LLM_MODEL     = "gpt-4o"

# --- Upload settings ---
UPLOAD_NOTE   = "Tagged by LLM"   # version note visible in Transkribus

## 3. Authenticate

In [3]:
import logging
from transkribus_api_client import TranskribusAPIClient
from ai_client import create_ai_client

logging.basicConfig(level=logging.WARNING)

# Authenticate with Transkribus
transkribus = TranskribusAPIClient(TRANSKRIBUS_USERNAME, TRANSKRIBUS_PASSWORD)
assert transkribus.authenticate(), "Transkribus authentication failed — check your credentials"
print("Authenticated with Transkribus")

# Set up LLM client
llm = create_ai_client(LLM_PROVIDER, api_key=LLM_API_KEY)
print(f"LLM client ready ({LLM_PROVIDER} / {LLM_MODEL})")

Authenticated with Transkribus
LLM client ready (openai / gpt-4o)


## 4. Download PAGE XML

In [8]:
page_xml = transkribus.get_page_xml(COLLECTION_ID, DOCUMENT_ID, PAGE_NR)
assert page_xml, "Failed to download PAGE XML — check your collection/document/page IDs"

print(f"Downloaded PAGE XML ({len(page_xml):,} characters)")
print("--- Preview (first 800 chars) ---")
print(page_xml[600:2000])

Downloaded PAGE XML (16,878 characters)
--- Preview (first 800 chars) ---
FOXVJ" imageId="87781625"/>
    </Metadata>
    <Page imageFilename="104611609.jpg" imageWidth="3744" imageHeight="5616">
        <ReadingOrder>
            <OrderedGroup id="ro_1770298547882" caption="Regions reading order">
                <RegionRefIndexed index="0" regionRef="tr_2"/>
                <RegionRefIndexed index="1" regionRef="tr_1"/>
                <RegionRefIndexed index="2" regionRef="tr_3"/>
            </OrderedGroup>
        </ReadingOrder>
        <TextRegion id="tr_2" custom="readingOrder {index:0;}">
            <Coords points="370,935 2802,935 2802,2423 370,2423"/>
            <TextLine id="tr_2_tl_1" custom="readingOrder {index:0;}">
                <Coords points="377,1060 471,1086 585,1027 699,1131 959,1043 1066,1138 1164,1053 1287,1027 1726,1034 1830,1092 1950,991 2194,1092 2470,1001 2632,1102 2766,1082 2769,868 2646,826 2431,881 2048,832 1823,868 1752,923 1628,845 1476,884 1238,809 

## 5. Process with LLM

Edit the prompt below to describe the tagging task you want the LLM to perform.

In [6]:
SYSTEM_PROMPT = """\
You are an expert in historical document analysis and TEI/PAGE XML markup.
Your task is to add named-entity tags to the transcribed text inside a PAGE XML document.

Rules:
- Identify persons, places, organisations, and dates in the TextLine content.
- Wrap recognised entities using Transkribus custom tags, e.g.:
    <custom type="named-entity" value="person" offset="0" length="5"/>
  added inside the <TextLine> element's <TextEquiv> or as a <Word> custom attribute.
- Do NOT change any coordinates, IDs, or structural elements.
- Return ONLY the complete, valid PAGE XML — no explanation, no markdown fences.
"""

USER_PROMPT = f"""\
Here is the PAGE XML to tag:

{page_xml}
"""

print("Sending to LLM…")
response = llm.prompt(LLM_MODEL, USER_PROMPT, system_prompt=SYSTEM_PROMPT)
tagged_xml = response.text

print(f"LLM processing complete ({response.usage.total_tokens:,} tokens used, {response.duration:.1f}s)")
print("--- Preview (first 800 chars) ---")
print(tagged_xml[600:1600])

Sending to LLM…
LLM processing complete (15,123 tokens used, 108.9s)
--- Preview (first 800 chars) ---
<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<PcGts xmlns="http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15">
    <Metadata>
        <Creator>prov=READ-COOP:name=PyLaia@TranskribusPlatform:version=2.38.1:model_id=504857:lm=none:date=05_02_2026_15:23</Creator>
        <Created>2025-10-20T16:01:55.944+02:00</Created>
        <LastChange>2026-02-05T15:23:50.087+01:00</LastChange>
        <TranskribusMetadata docId="11509276" pageId="104611609" pageNr="55" status="IN_PROGRESS" userId="9987" imgUrl="https://files.transkribus.eu/Get?fileType=view&amp;id=UJXCXZOIGQXZMWYQBKGFOXVJ" imageId="87781625"/>
    </Metadata>
    <Page imageFilename="104611609.jpg" imageWidth="3744" imageHeight="5616">
        <ReadingOrder>
            <OrderedGroup id="ro_1770298547882" caption=


## 6. Upload result back to Transkribus

In [7]:
success = transkribus.upload_page_transcript(
    collection_id=COLLECTION_ID,
    doc_id=DOCUMENT_ID,
    page_nr=PAGE_NR,
    xml_content=tagged_xml,
    status="IN_PROGRESS",
    note=UPLOAD_NOTE,
)

if success:
    print(f"Upload successful! Open Transkribus to review page {PAGE_NR} of document {DOCUMENT_ID}.")
else:
    print("Upload failed — check the error messages above.")

Upload successful! Open Transkribus to review page 55 of document 11509276.
